In [4]:
# ============================================================
# IND MODEL — Trend Prediction (STRICT MODEL-CONSISTENT)
#   - two largest components
#   - scaled global trend
#   - NO trend wrapping
# ============================================================

import numpy as np
import pyreadr
import geopandas as gpd
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from tqdm import tqdm
from pathlib import Path

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
BASE_DIR = Path(r"D:\77\Research\temp\snow")
DIST_TH = 0.22
period = 52

no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

# ------------------------------------------------------------
# 1️⃣ Load data
# ------------------------------------------------------------
snow = pyreadr.read_r(BASE_DIR/"snow_cleaned_full.Rda")
snow = list(snow.values())[0].reset_index(drop=True)
snow = snow.drop(index=no_nbs).reset_index(drop=True)

coords_full = snow.iloc[:, :2].to_numpy()
y_full = snow.iloc[:, 2:].to_numpy()

# ------------------------------------------------------------
# 2️⃣ Keep TWO largest connected components
# ------------------------------------------------------------
gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(coords_full[:,0], coords_full[:,1]),
    crs="EPSG:4326"
).to_crs("+proj=aeqd +lat_0=90 +lon_0=-100")

xy = np.vstack([gdf.geometry.x, gdf.geometry.y]).T / 1e6
W = (squareform(pdist(xy)) <= DIST_TH).astype(int)
np.fill_diagonal(W, 0)
W = csr_matrix(W)

n_comp, labels = connected_components(W, directed=False)
sizes = np.bincount(labels)
order = np.argsort(sizes)[::-1]

keep = np.sort(np.concatenate([
    np.where(labels == order[0])[0],
    np.where(labels == order[1])[0]
]))

coords = coords_full[keep]
y = y_full[keep]

S, TT = y.shape
print("Using S =", S, "TT =", TT)

# ------------------------------------------------------------
# 3️⃣ Global trend scaling (HISTORICAL ONLY)
# ------------------------------------------------------------
t_hist = np.arange(1, TT+1)
mean_hist = t_hist.mean()
sd_hist   = t_hist.std(ddof=0)

# ------------------------------------------------------------
# 4️⃣ Load posterior
# ------------------------------------------------------------
res01 = np.load(BASE_DIR/"ind01.npz")
res10 = np.load(BASE_DIR/"ind10.npz")

theta01 = res01["all_theta"]   # (4S , M)
theta10 = res10["all_theta"]

M = theta01.shape[1]
print("Loaded posterior samples M =", M)

# ------------------------------------------------------------
# 5️⃣ Slice blocks
# ------------------------------------------------------------
beta0_01 = theta01[0*S:1*S, :]
beta1_01 = theta01[1*S:2*S, :]
beta2_01 = theta01[2*S:3*S, :]
alpha_01 = theta01[3*S:4*S, :]

beta0_10 = theta10[0*S:1*S, :]
beta1_10 = theta10[1*S:2*S, :]
beta2_10 = theta10[2*S:3*S, :]
alpha_10 = theta10[3*S:4*S, :]

# ------------------------------------------------------------
# 6️⃣ Allocate
# ------------------------------------------------------------
weekly_ini   = np.zeros((M, S, 52, 2))
weekly_final = np.zeros((M, S, 52, 2))

inv_logit = lambda x: 1 / (1 + np.exp(-x))

print("Begin IND trend prediction (strict)...")

# ============================================================
# MAIN LOOP
# ============================================================

for m in tqdm(range(M), desc="Posterior samples"):

    b0  = beta0_01[:, m]
    b1  = beta1_01[:, m]
    b2  = beta2_01[:, m]
    a01 = alpha_01[:, m]

    b0s = beta0_10[:, m]
    b1s = beta1_10[:, m]
    b2s = beta2_10[:, m]
    a10 = alpha_10[:, m]

    init_state = np.column_stack([
        y[:,0] == 0,
        y[:,0] == 1
    ]).astype(float)

    # =========================================================
    # FIRST YEAR
    # =========================================================
    for week_idx in range(1, 53):

        curr = init_state.copy()

        if week_idx > 1:
            for t in range(1, week_idx):

                t_scaled = (t - mean_hist) / sd_hist

                eta01 = (
                    b0
                    + b1*np.cos(2*np.pi*t/period)
                    + b2*np.sin(2*np.pi*t/period)
                    + a01*t_scaled
                )

                eta10 = (
                    b0s
                    + b1s*np.cos(2*np.pi*t/period)
                    + b2s*np.sin(2*np.pi*t/period)
                    + a10*t_scaled
                )

                p01 = inv_logit(eta01)
                p10 = inv_logit(eta10)

                c0 = curr[:,0]
                c1 = curr[:,1]

                curr = np.column_stack([
                    c0*(1-p01) + c1*p10,
                    c0*p01     + c1*(1-p10)
                ])

        weekly_ini[m,:,week_idx-1,:] = curr

    # =========================================================
    # FINAL YEAR (52 years ahead)
    # =========================================================
    curr = init_state.copy()
    total_steps = 52*52

    for t in range(1, total_steps+1):

        t_scaled = (t - mean_hist) / sd_hist

        eta01 = (
            b0
            + b1*np.cos(2*np.pi*t/period)
            + b2*np.sin(2*np.pi*t/period)
            + a01*t_scaled
        )

        eta10 = (
            b0s
            + b1s*np.cos(2*np.pi*t/period)
            + b2s*np.sin(2*np.pi*t/period)
            + a10*t_scaled
        )

        p01 = inv_logit(eta01)
        p10 = inv_logit(eta10)

        c0 = curr[:,0]
        c1 = curr[:,1]

        curr = np.column_stack([
            c0*(1-p01) + c1*p10,
            c0*p01     + c1*(1-p10)
        ])

        # store only last year
        if t > 51*52:
            w = t - 51*52 - 1
            weekly_final[m,:,w,:] = curr

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------
np.savez_compressed(
    BASE_DIR/"predict_ind.npz",
    weekly_ini=weekly_ini,
    weekly_final=weekly_final
)

print("IND prediction finished.")

Using S = 1557 TT = 2704
Loaded posterior samples M = 1000
Begin IND trend prediction (strict)...


Posterior samples: 100%|██████████| 1000/1000 [06:17<00:00,  2.65it/s]


IND prediction finished (strict version).


In [7]:
# ============================================================
# BYM WEEKLY MODEL — Trend Prediction (STRICT MODEL-CONSISTENT)
#   - two largest components
#   - scaled global trend
#   - weekly tau
#   - NO trend wrapping
# ============================================================

import numpy as np
import pyreadr
import geopandas as gpd
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from tqdm import tqdm
from pathlib import Path
import pickle

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
BASE_DIR = Path(r"D:\77\Research\temp\snow")
DIST_TH = 0.22
period = 52

no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

# ------------------------------------------------------------
# 1️⃣ Load data
# ------------------------------------------------------------
snow = pyreadr.read_r(BASE_DIR/"snow_cleaned_full.Rda")
snow = list(snow.values())[0].reset_index(drop=True)
snow = snow.drop(index=no_nbs).reset_index(drop=True)

coords_full = snow.iloc[:, :2].to_numpy()
y_full = snow.iloc[:, 2:].to_numpy()

# ------------------------------------------------------------
# 2️⃣ Keep TWO largest connected components
# ------------------------------------------------------------
gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(coords_full[:,0], coords_full[:,1]),
    crs="EPSG:4326"
).to_crs("+proj=aeqd +lat_0=90 +lon_0=-100")

xy = np.vstack([gdf.geometry.x, gdf.geometry.y]).T / 1e6
W = (squareform(pdist(xy)) <= DIST_TH).astype(int)
np.fill_diagonal(W, 0)
W = csr_matrix(W)

n_comp, labels = connected_components(W, directed=False)
sizes = np.bincount(labels)
order = np.argsort(sizes)[::-1]

keep = np.sort(np.concatenate([
    np.where(labels == order[0])[0],
    np.where(labels == order[1])[0]
]))

coords = coords_full[keep]
y = y_full[keep]

S, TT = y.shape
print("Using S =", S, "TT =", TT)

# ------------------------------------------------------------
# 3️⃣ Historical trend scaling
# ------------------------------------------------------------
t_hist = np.arange(1, TT+1)
mean_hist = t_hist.mean()
sd_hist   = t_hist.std(ddof=0)

# ------------------------------------------------------------
# 4️⃣ Load posterior
# ------------------------------------------------------------
with open(BASE_DIR/"bym01_weekly.pkl", "rb") as f:
    res01 = pickle.load(f)

with open(BASE_DIR/"bym10_weekly.pkl", "rb") as f:
    res10 = pickle.load(f)

all_eta01 = res01["all_eta"]   # (8S , M)
all_tau01 = res01["all_tau"]   # (8*52 , M)

all_eta10 = res10["all_eta"]
all_tau10 = res10["all_tau"]

M = all_eta01.shape[1]
print("Loaded posterior samples M =", M)

K_total = 8

# ------------------------------------------------------------
# 5️⃣ Allocate
# ------------------------------------------------------------
weekly_ini   = np.zeros((M, S, 52, 2))
weekly_final = np.zeros((M, S, 52, 2))

inv_logit = lambda x: 1 / (1 + np.exp(-x))

print("Begin BYM weekly trend prediction (strict)...")

# ============================================================
# MAIN LOOP
# ============================================================

for m in tqdm(range(M), desc="Posterior samples"):

    eta01 = all_eta01[:, m]
    tau01 = all_tau01[:, m]

    eta10 = all_eta10[:, m]
    tau10 = all_tau10[:, m]

    # reshape
    eta01_mat = eta01.reshape(K_total, S)
    tau01_mat = tau01.reshape(K_total, 52)

    eta10_mat = eta10.reshape(K_total, S)
    tau10_mat = tau10.reshape(K_total, 52)

    init_state = np.column_stack([
        y[:,0] == 0,
        y[:,0] == 1
    ]).astype(float)

    # =========================================================
    # FIRST YEAR
    # =========================================================
    for week_idx in range(1, 53):

        curr = init_state.copy()

        if week_idx > 1:
            for t in range(1, week_idx):

                t_scaled = (t - mean_hist) / sd_hist
                w = (t-1) % 52

                x_vec = np.array([
                    1.0,
                    np.cos(2*np.pi*t/period),
                    np.sin(2*np.pi*t/period),
                    t_scaled
                ])

                # duplicate for CAR+IID structure
                x_full = np.repeat(x_vec, 2)

                psi01 = np.zeros(S)
                psi10 = np.zeros(S)

                for k in range(K_total):
                    psi01 += x_full[k] * eta01_mat[k] * tau01_mat[k, w]
                    psi10 += x_full[k] * eta10_mat[k] * tau10_mat[k, w]

                p01 = inv_logit(psi01)
                p10 = inv_logit(psi10)

                c0 = curr[:,0]
                c1 = curr[:,1]

                curr = np.column_stack([
                    c0*(1-p01) + c1*p10,
                    c0*p01     + c1*(1-p10)
                ])

        weekly_ini[m,:,week_idx-1,:] = curr

    # =========================================================
    # FINAL YEAR (52 years ahead)
    # =========================================================
    curr = init_state.copy()
    total_steps = 52*52

    for t in range(1, total_steps+1):

        t_scaled = (t - mean_hist) / sd_hist
        w = (t-1) % 52

        x_vec = np.array([
            1.0,
            np.cos(2*np.pi*t/period),
            np.sin(2*np.pi*t/period),
            t_scaled
        ])

        x_full = np.repeat(x_vec, 2)

        psi01 = np.zeros(S)
        psi10 = np.zeros(S)

        for k in range(K_total):
            psi01 += x_full[k] * eta01_mat[k] * tau01_mat[k, w]
            psi10 += x_full[k] * eta10_mat[k] * tau10_mat[k, w]

        p01 = inv_logit(psi01)
        p10 = inv_logit(psi10)

        c0 = curr[:,0]
        c1 = curr[:,1]

        curr = np.column_stack([
            c0*(1-p01) + c1*p10,
            c0*p01     + c1*(1-p10)
        ])

        if t > 51*52:
            w_store = t - 51*52 - 1
            weekly_final[m,:,w_store,:] = curr

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------
np.savez_compressed(
    BASE_DIR/"trend_bym_weekly.npz",
    weekly_ini=weekly_ini,
    weekly_final=weekly_final
)

print("BYM weekly prediction finished.")

Using S = 1557 TT = 2704
Loaded posterior samples M = 1000
Begin BYM weekly trend prediction (strict)...


Posterior samples: 100%|██████████| 1000/1000 [11:52<00:00,  1.40it/s]


BYM weekly prediction finished.


In [8]:
# ============================================================
# BYM WEEKLY + FACTOR (interaction-only)
# STRICT MODEL-CONSISTENT TREND PREDICTION
# ============================================================

import numpy as np
import pyreadr
import geopandas as gpd
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from tqdm import tqdm
from pathlib import Path
import pickle
import pandas as pd

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
BASE_DIR = Path(r"D:\77\Research\temp\snow")
DIST_TH = 0.22
period = 52

no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

# ------------------------------------------------------------
# 1️⃣ Load snow data
# ------------------------------------------------------------
snow = pyreadr.read_r(BASE_DIR/"snow_cleaned_full.Rda")
snow = list(snow.values())[0].reset_index(drop=True)
snow = snow.drop(index=no_nbs).reset_index(drop=True)

coords_full = snow.iloc[:, :2].to_numpy()
y_full = snow.iloc[:, 2:].to_numpy()

# ------------------------------------------------------------
# 2️⃣ Keep TWO largest connected components
# ------------------------------------------------------------
gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(coords_full[:,0], coords_full[:,1]),
    crs="EPSG:4326"
).to_crs("+proj=aeqd +lat_0=90 +lon_0=-100")

xy = np.vstack([gdf.geometry.x, gdf.geometry.y]).T / 1e6
W = (squareform(pdist(xy)) <= DIST_TH).astype(int)
np.fill_diagonal(W, 0)
W = csr_matrix(W)

n_comp, labels = connected_components(W, directed=False)
sizes = np.bincount(labels)
order = np.argsort(sizes)[::-1]

keep = np.sort(np.concatenate([
    np.where(labels == order[0])[0],
    np.where(labels == order[1])[0]
]))

coords = coords_full[keep]
y = y_full[keep]

S, TT = y.shape
print("Using S =", S)

# ------------------------------------------------------------
# 3️⃣ Historical trend scaling
# ------------------------------------------------------------
t_hist = np.arange(1, TT+1)
mean_hist = t_hist.mean()
sd_hist   = t_hist.std(ddof=0)

# ------------------------------------------------------------
# 4️⃣ Load covariates (scaled exactly as fitting)
# ------------------------------------------------------------
# latitude
lat_raw = coords[:,1]
lat = (lat_raw - lat_raw.mean()) / lat_raw.std()

# elevation
elev_raw = pd.read_csv(BASE_DIR/"curr_elev.csv").iloc[:,3].to_numpy()
elev = (elev_raw[keep] - elev_raw[keep].mean()) / elev_raw[keep].std()

# temperature (global scaling as in MCMC)
snow_temp = pyreadr.read_r(BASE_DIR/"snow_temp_full.Rda")
snow_temp = list(snow_temp.values())[0].reset_index(drop=True)
temp_full = snow_temp.drop(index=no_nbs).iloc[:,2:].to_numpy()
temp_full = temp_full[keep]
temp_scaled = (temp_full - temp_full.mean()) / temp_full.std()

# ------------------------------------------------------------
# 5️⃣ Load posterior
# ------------------------------------------------------------
with open(BASE_DIR/"bym01_cov_weekly_tau9.pkl","rb") as f:
    res01 = pickle.load(f)

with open(BASE_DIR/"bym10_cov_weekly_tau9.pkl","rb") as f:
    res10 = pickle.load(f)

all_eta01 = res01["all_eta"]   # (8S+3 , M)
all_tau01 = res01["all_tau"]   # (8*52 , M)

all_eta10 = res10["all_eta"]
all_tau10 = res10["all_tau"]

M = all_eta01.shape[1]
print("Loaded posterior samples M =", M)

K_base = 4
K_total = 8

# ------------------------------------------------------------
# 6️⃣ Allocate
# ------------------------------------------------------------
weekly_ini   = np.zeros((M, S, 52, 2))
weekly_final = np.zeros((M, S, 52, 2))

inv_logit = lambda x: 1 / (1 + np.exp(-x))

print("Begin BYM weekly + factor trend prediction (strict)...")

# ============================================================
# MAIN LOOP
# ============================================================

for m in tqdm(range(M), desc="Posterior samples"):

    eta01 = all_eta01[:, m]
    tau01 = all_tau01[:, m]

    eta10 = all_eta10[:, m]
    tau10 = all_tau10[:, m]

    # spatial blocks
    eta01_sp = eta01[:K_total*S].reshape(K_total, S)
    eta10_sp = eta10[:K_total*S].reshape(K_total, S)

    tau01_mat = tau01.reshape(K_total, 52)
    tau10_mat = tau10.reshape(K_total, 52)

    # gamma (interaction-only)
    gamma01 = eta01[K_total*S:]
    gamma10 = eta10[K_total*S:]

    init_state = np.column_stack([
        y[:,0] == 0,
        y[:,0] == 1
    ]).astype(float)

    # =========================================================
    # FIRST YEAR
    # =========================================================
    for week_idx in range(1, 53):

        curr = init_state.copy()

        if week_idx > 1:
            for t in range(1, week_idx):

                t_scaled = (t - mean_hist) / sd_hist
                w = (t-1) % 52

                x_vec = np.array([
                    1.0,
                    np.cos(2*np.pi*t/period),
                    np.sin(2*np.pi*t/period),
                    t_scaled
                ])

                x_full = np.repeat(x_vec, 2)

                psi01 = np.zeros(S)
                psi10 = np.zeros(S)

                for k in range(K_total):
                    psi01 += x_full[k] * eta01_sp[k] * tau01_mat[k, w]
                    psi10 += x_full[k] * eta10_sp[k] * tau10_mat[k, w]

                # interaction-only factor
                temp_t = temp_scaled[:, (t-1) % TT]

                psi01 += t_scaled * (
                    gamma01[0] * lat
                    + gamma01[1] * elev
                    + gamma01[2] * temp_t
                )

                psi10 += t_scaled * (
                    gamma10[0] * lat
                    + gamma10[1] * elev
                    + gamma10[2] * temp_t
                )

                p01 = inv_logit(psi01)
                p10 = inv_logit(psi10)

                c0 = curr[:,0]
                c1 = curr[:,1]

                curr = np.column_stack([
                    c0*(1-p01) + c1*p10,
                    c0*p01     + c1*(1-p10)
                ])

        weekly_ini[m,:,week_idx-1,:] = curr

    # =========================================================
    # FINAL YEAR
    # =========================================================
    curr = init_state.copy()
    total_steps = 52*52

    for t in range(1, total_steps+1):

        t_scaled = (t - mean_hist) / sd_hist
        w = (t-1) % 52

        x_vec = np.array([
            1.0,
            np.cos(2*np.pi*t/period),
            np.sin(2*np.pi*t/period),
            t_scaled
        ])

        x_full = np.repeat(x_vec, 2)

        psi01 = np.zeros(S)
        psi10 = np.zeros(S)

        for k in range(K_total):
            psi01 += x_full[k] * eta01_sp[k] * tau01_mat[k, w]
            psi10 += x_full[k] * eta10_sp[k] * tau10_mat[k, w]

        temp_t = temp_scaled[:, (t-1) % TT]

        psi01 += t_scaled * (
            gamma01[0] * lat
            + gamma01[1] * elev
            + gamma01[2] * temp_t
        )

        psi10 += t_scaled * (
            gamma10[0] * lat
            + gamma10[1] * elev
            + gamma10[2] * temp_t
        )

        p01 = inv_logit(psi01)
        p10 = inv_logit(psi10)

        c0 = curr[:,0]
        c1 = curr[:,1]

        curr = np.column_stack([
            c0*(1-p01) + c1*p10,
            c0*p01     + c1*(1-p10)
        ])

        if t > 51*52:
            w_store = t - 51*52 - 1
            weekly_final[m,:,w_store,:] = curr

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------
np.savez_compressed(
    BASE_DIR/"trend_weekly_bym+cov.npz",
    weekly_ini=weekly_ini,
    weekly_final=weekly_final
)

print("BYM weekly + factor prediction finished (strict).")

Using S = 1557
Loaded posterior samples M = 1000
Begin BYM weekly + factor trend prediction (strict)...


Posterior samples: 100%|██████████| 1000/1000 [14:58<00:00,  1.11it/s]


BYM weekly + factor prediction finished (strict).


In [10]:
import numpy as np
from pathlib import Path

import rpy2.robjects as ro
from rpy2.robjects import numpy2ri
from rpy2.robjects.conversion import localconverter


def predict_npz_to_rda(npz_path):
    """
    Convert any npz -> Rda
    Keeps all variable names as-is.
    """

    npz_path = Path(npz_path)
    rda_path = npz_path.with_suffix(".Rda")

    data = np.load(npz_path, allow_pickle=True)

    with localconverter(ro.default_converter + numpy2ri.converter):
        for k in data.files:
            obj = data[k]
            if obj.dtype == object:
                obj = obj.tolist()
            ro.globalenv[k] = obj

    # dynamically construct save command
    var_string = ", ".join(data.files)
    ro.r(f'save({var_string}, file="{rda_path.as_posix()}")')

    print(f"Saved {rda_path}")
# ------------------------------------------------------------
# CONVERT ALL TREND FILES TO Rda
# ------------------------------------------------------------

base = Path(r"D:/77/Research/temp/snow")

predict_npz_to_rda(base / "trend_ind.npz")
predict_npz_to_rda(base / "trend_bym_weekly.npz")
predict_npz_to_rda(base / "trend_weekly_bym+cov.npz")

Saved D:\77\Research\temp\snow\trend_ind.Rda


KeyboardInterrupt: 